# Lab 09: Assignment 2 tutorial

**Build it. Check it. Explain the difference.**

Open [Assignment 2](https://jduarte.physics.ucsd.edu/phys142/assignment2.pdf) beside this notebook.
We'll practice with a **harmonic oscillator**, then you'll apply the methods to the
**double well** in the homework.

**Assignment 2 is due Monday, November 2, per the syllabus.**

PHYS 142/242 · October 27, 2026

## Running and presenting

Use Python 3 on DataHub, or save your own copy in Colab. Run the cells in order.
If an import fails on DataHub, ask me. In Colab or your local environment, install
`numpy scipy matplotlib` with `%pip install numpy scipy matplotlib` if needed.

For RISE, open **Reveal Slideshow**. Use the right arrow to advance and
**Shift+Enter** to run a selected code cell. Reading notes marked **Skip** remain
available in ordinary notebook view. For a rehearsal, restart and Run All there,
then open the slideshow. The distributed notebook has no saved outputs.

Suggested 80-minute route: setup and B (20 min), A (20), C (10), D (15),
export checks (5), and your own assignment code/questions (10).

## What carries over to the homework?

| Part | What we'll practice |
| :--- | :--- |
| B, first | Find eigenstates; check normalization and convergence |
| A | Build the midpoint propagator; measure motion |
| C | Test approximate states with the same Hamiltonian |
| D | Compare two methods from the same initial state |

**Keep the plots and numbers tied to the code that produced them.**

## The homework system

$$
V(x)=\alpha x^4-2x^2+\frac{1}{\alpha},
\qquad \alpha=0.3+\frac{d}{5000}.
$$

**Use the last three TSN digits for $d$, including leading zeros.**

The minima are at $\pm1/\sqrt{\alpha}$; the barrier is $1/\alpha$.
Sketch what changes when $\alpha$ increases.

We'll use $m=\hbar=1$ throughout.

### Keep these homework conventions beside your code

- **Grid:** 600 intervals, **601 points**, from $-4$ to $4$; $\Delta x=8/600$.
- **Time:** $\Delta t=\pi/128$; build times as integer step numbers times $\Delta t$.
- **Integrals:** $\Delta x\sum_i$, including both endpoints, for every norm, overlap,
  expectation value, and propagation step. Use this rule consistently.
- **Hamiltonian:** retain all 601 unknowns. The missing neighbors outside the box
  are zero; the endpoint unknowns themselves are retained.
- **A starts from the right-well Gaussian. D starts from the two lowest eigenstates.**

The worked oscillator below uses the same base grid and time step. It has a
different potential and different numerical answers. B's convergence tests are
the only homework calculations that change the base grid.

## Imports first

**NumPy for arrays. SciPy for the sparse eigenproblem. Matplotlib for plots.**

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import eigsh

plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

## One grid, two numerical methods

**Check the number of points and the spacing before doing physics.**

In [ ]:
ND = 600
x = np.linspace(-4.0, 4.0, ND + 1)
dx = x[1] - x[0]
dt = np.pi / 128
omega = 2.0

def V_demo(position):
    return 0.5 * omega**2 * position**2

assert x.size == 601 and np.isclose(dx, 8 / 600)
print(f"{x.size} positions; dx = {dx:.6f}; dt = {dt:.6f}")

## Normalize on the grid

**An eigensolver normalizes an array. We need a spatial integral.**

$$
\langle a|b\rangle\approx\Delta x\sum_i a_i^*b_i.
$$

`np.vdot` conjugates its first argument.

In [ ]:
def inner(left, right, spacing):
    return spacing * np.vdot(left, right)

def normalize(state, spacing):
    norm = inner(state, state, spacing).real
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError("The state must have a finite, positive norm.")
    return state / np.sqrt(norm)

## B: Turn derivatives into a matrix

**Three nonzero diagonals. Keep the endpoints.**

$$
H_{ii}=\frac{1}{\Delta x^2}+V(x_i),\qquad
H_{i,i\pm1}=-\frac{1}{2\Delta x^2}.
$$

What terms are present in the first row? Which neighbor is missing?

In [ ]:
def hamiltonian(grid, potential):
    spacing = grid[1] - grid[0]
    diagonal = 1 / spacing**2 + potential(grid)
    neighbor = np.full(grid.size - 1, -0.5 / spacing**2)
    return diags([neighbor, diagonal, neighbor], [-1, 0, 1], format="csc")

H = hamiltonian(x, V_demo)
print("Shape:", H.shape, "Stored entries:", H.nnz)
print(H[:3, :3].toarray())

## Find the two lowest states

**Shift-invert targets eigenvalues near $\sigma$.**

Both potentials are nonnegative, so $\sigma=0$ sits below the spectrum.
With shift-invert, `which="LM"` selects the largest transformed magnitudes.

In [ ]:
def lowest_pair(matrix):
    energies, vectors = eigsh(
        matrix, k=2, sigma=0.0, which="LM",
        v0=np.linspace(1.0, 2.0, matrix.shape[0]),
    )
    order = np.argsort(energies)
    return energies[order], vectors[:, order]

energies, raw_phi = lowest_pair(H)
phi = np.column_stack([normalize(raw_phi[:, j], dx) for j in range(2)])

## Choose signs, then check

**A sign flip is allowed. Relative signs determine a superposition.**

Choose the ground state positive and the first excited state's right lobe positive.

In [ ]:
phi[:, 0] *= np.sign(phi[np.argmax(np.abs(phi[:, 0])), 0])
right = np.flatnonzero(x > 0)
right_peak = right[np.argmax(np.abs(phi[right, 1]))]
phi[:, 1] *= np.sign(phi[right_peak, 1])

print("Grid overlap matrix:")
print(dx * phi.T @ phi)
for j in range(2):
    residual = H @ phi[:, j] - energies[j] * phi[:, j]
    print(f"State {j}: residual norm = {np.linalg.norm(residual):.2e}")

## Check an answer we know

**Oscillator energies: $E_n=(n+\tfrac12)\omega$.**

A small residual checks the matrix equation. Agreement with physics checks
whether we built the intended matrix.

In [ ]:
exact_energies = omega * (np.arange(2) + 0.5)
print("Computed:", energies)
print("Exact:   ", exact_energies)
print("Errors:  ", energies - exact_energies)

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True, layout="constrained")
axes[0].plot(x, V_demo(x), color="black")
axes[0].set(ylabel="Potential energy", ylim=(0, 6))
for j in range(2):
    axes[1].plot(x, phi[:, j], label=f"State {j}")
axes[1].set(xlabel="Position x", ylabel="Wavefunction")
axes[1].legend()
plt.show()

## Change one numerical choice at a time

- **Half the spacing:** same box, more points.
- **Larger box:** same spacing, more points.

**Rebuild the Hamiltonian for each grid.**

In [ ]:
demo_gap = energies[1] - energies[0]
grids = [("Original", -4, 4, 600), ("Half spacing", -4, 4, 1200),
         ("Larger box", -6, 6, 900)]
convergence = []
for label, left_edge, right_edge, intervals in grids:
    grid = np.linspace(left_edge, right_edge, intervals + 1)
    pair, _ = lowest_pair(hamiltonian(grid, V_demo))
    gap = pair[1] - pair[0]
    convergence.append((label, pair[0], pair[1], gap, gap - demo_gap))
print(f"{'Grid':14s} {'E0':>12s} {'E1':>12s} {'Gap':>12s} {'Gap change':>13s}")
for label, e0, e1, gap, change in convergence:
    print(f"{label:14s} {e0:12.8f} {e1:12.8f} {gap:12.8f} {change:13.2e}")

### Your turn: start B in your own notebook

1. Implement the double-well potential and check its minima and barrier by hand.
2. Build the Hamiltonian. Find and normalize the two lowest states with shift-invert.
   Check their signs, parity, overlap, and residuals.
3. Before using your TSN value, validate at $\alpha=0.4,0.6$ against
   Banerjee and Bhatnagar's Table I, linked in the assignment and shown in Lecture 11.
   Their $\lambda=\alpha/4$. Report the digits that agree.
4. At your own $\alpha$, use the three grids above. Tabulate both energies,
   the gap, and its change. Which test isolates spacing? Which isolates the box?
5. Keep the **original-grid** arrays and energies under distinct names for later
   calculations and export. Don't leave the last convergence grid in `x` or `H`.

For the homework figure, put the potential in a separate panel from the two
eigenfunctions. A tiny solver residual alone does not establish grid convergence.

## A: One short-time propagation step

**A row is a destination. A column is a source.**

$$
K_{ba}=\exp\!\left[i\left(\frac{(x_b-x_a)^2}{2\Delta t}
-V\!\left(\frac{x_a+x_b}{2}\right)\Delta t\right)\right].
$$

Use the midpoint potential, multiply by $\Delta x$, then normalize.

In [ ]:
destination = x[:, None]
source = x[None, :]
midpoint = (destination + source) / 2
K_demo = np.exp(1j * (
    (destination - source)**2 / (2 * dt) - V_demo(midpoint) * dt
))
U_demo = dx * K_demo
print("Kernel shape and dtype:", K_demo.shape, K_demo.dtype)

## Start away from the center

**Predict:** where will the mean position go first?

This is a displaced oscillator Gaussian. Its center should oscillate.

In [ ]:
demo_initial = normalize(np.exp(-omega * (x - 1.0)**2 / 2), dx)
demo_period = 2 * np.pi / demo_gap
steps = int(np.ceil(demo_period / dt))
t = np.arange(steps + 1) * dt

print("Initial norm:", inner(demo_initial, demo_initial, dx).real)
print("Initial mean:", dx * np.sum(x * np.abs(demo_initial)**2))
print(f"{steps} steps; final time = {t[-1]:.6f}")

## Take the steps

**Store the initial state at index zero. Normalize every update.**

In [ ]:
def propagate(update, initial, n_steps, spacing):
    states = np.empty((n_steps + 1, initial.size), dtype=np.complex128)
    states[0] = normalize(initial, spacing)
    for n in range(n_steps):
        states[n + 1] = normalize(update @ states[n], spacing)
    return states

gaussian_states = propagate(U_demo, demo_initial, steps, dx)
gaussian_density = np.abs(gaussian_states)**2
gaussian_mean = dx * (gaussian_density @ x)
norms = dx * gaussian_density.sum(axis=1)
print("Largest norm error:", np.max(np.abs(norms - 1)))

## Read the motion before choosing an index

**Find the broad right-to-left transfer first. Then inspect that window.**

A first zero crossing is halfway to maximum leftward displacement.
A global minimum can select a later cycle; a small local minimum can select a ripple.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4), layout="constrained")
ax.plot(t, gaussian_mean, label="Midpoint propagation")
ax.plot(t, np.cos(omega * t), "--", label="Exact oscillator center")
ax.axhline(0, color="gray", linewidth=0.7)
ax.set(xlabel="Time", ylabel="Mean position")
ax.legend()
plt.show()

## Measure within the first broad trough

**For this demo**, inspect the window from 0.3 to 0.7 of its period.
Check the chosen point against the curve.

In [ ]:
window = np.flatnonzero((t >= 0.3 * demo_period) & (t <= 0.7 * demo_period))
minimum_index = window[np.argmin(gaussian_mean[window])]
fig, ax = plt.subplots(figsize=(7, 3.2), layout="constrained")
ax.plot(t, gaussian_mean)
ax.plot(t[minimum_index], gaussian_mean[minimum_index], "o", color="tab:red")
ax.axvspan(t[window[0]], t[window[-1]], color="gray", alpha=0.12)
ax.set(xlabel="Time", ylabel="Mean position")
plt.show()
print(f"Selected minimum: t={t[minimum_index]:.6f}, mean={gaussian_mean[minimum_index]:.6f}")

### Your turn: transfer A to the double well

- Construct the **right-well Gaussian** $\psi_+$ specified in the assignment.
  Its harmonic frequency is $2\sqrt{2}$. Normalize on the original grid.
- Build the midpoint kernel with your potential. Do not use the exact oscillator
  kernel from Assignment 1. Check that your arrays have the intended row/column order.
- Use B's gap to estimate a full period $2\pi/\Delta E$, then inspect your curve
  for the slow trip right → left → right. Run at least through $16\pi$ for export,
  and longer if the slow return needs it.
- Mark the **first major minimum** and give its time and mean position. Identify a
  faster fluctuation if resolved. Choose a window from your curve; our oscillator
  window is not an automatic minimum finder for the homework.
- Compare that measured time with D later. Do not replace the measurement with
  $\pi/\Delta E$. The Gaussian can contain more than two eigenstates.

The time grid limits how finely you locate a minimum. Discuss ambiguous ripples
or a broad trough rather than reporting every printed decimal as measured precision.

## C: Test an approximate state

**Keep the denominator. The trial state may be unnormalized.**

$$
E[\chi]=\frac{\langle\chi|H|\chi\rangle}{\langle\chi|\chi\rangle}.
$$

Predict what happens when we multiply $\chi$ by 3.

In [ ]:
def rayleigh(state, matrix, spacing):
    return (inner(state, matrix @ state, spacing)
            / inner(state, state, spacing)).real

width = 0.7 * omega
trial = np.exp(-width * x**2 / 2)
trial_energy = rayleigh(trial, H, dx)
scaled_energy = rayleigh(3 * trial, H, dx)
print("Trial norm:", inner(trial, trial, dx).real)
print("Energy, before/after rescaling:", trial_energy, scaled_energy)
print("Ground-state energy:", energies[0])

## A small energy error can be a large gap error

**Subtract the signed errors.**

$$
(E_{\mathrm A}-E_{\mathrm S})-(E_1-E_0)
=(E_{\mathrm A}-E_1)-(E_{\mathrm S}-E_0).
$$

Made-up example: exact energies 1.00, 1.02; estimates 1.01, 1.04.
What are the relative errors in the energies and in their gap?

### Your turn: C needs the same Hamiltonian as B

1. Build $\psi_\mathrm S,\psi_\mathrm A$ from the assignment's two local Gaussians.
   Their overlap means the combinations are not exactly normalized.
2. Calculate both Rayleigh quotients using B's original-grid Hamiltonian. Report
   relative errors in both energies and their gap, plus the signed energy errors.
3. Normalize $\psi_\mathrm S$. Overlay $|\psi_\mathrm S|$ and $|\phi_0|$ on a
   logarithmic vertical scale. Mark an actual discrepancy in the barrier region.
4. Use those plots and signed errors to explain the gap error. An accurate shape
   near a minimum does not establish an accurate wavefunction in the barrier.

## D: Start both methods from the same state

**A different initial state gives a different comparison.**

We'll combine the two oscillator eigenstates, using the signs we checked earlier.

In [ ]:
two_state_initial = normalize((phi[:, 0] + phi[:, 1]) / np.sqrt(2), dx)
two_state_pi = propagate(U_demo, two_state_initial, steps, dx)
phases = np.exp(-1j * t[:, None] * energies[None, :])
two_state_eig = phases @ phi.T / np.sqrt(2)

mean_pi = dx * (np.abs(two_state_pi)**2 @ x)
mean_eig = dx * (np.abs(two_state_eig)**2 @ x)

## Put a number on the disagreement

**Use every time step in the comparison interval.**

The final stored step reaches or passes one period. Compare only times at or
before the period when reporting the maximum.

In [ ]:
in_period = np.flatnonzero(t <= demo_period)
differences = np.abs(mean_pi - mean_eig)
worst = in_period[np.argmax(differences[in_period])]
fig, ax = plt.subplots(figsize=(7, 3.3), layout="constrained")
ax.plot(t, mean_pi, label="Midpoint propagation")
ax.plot(t, mean_eig, "--", label="Eigenstates")
ax.plot(t[worst], mean_pi[worst], "o", color="tab:red")
ax.set(xlabel="Time", ylabel="Mean position")
ax.legend()
plt.show()
print(f"Max difference={differences[worst]:.3e} at t={t[worst]:.6f}")
print("Means at that time:", mean_pi[worst], mean_eig[worst])

## Compare the densities too

**Same grid. Same actual time. Both methods in each panel.**

In [ ]:
requested = np.array([0.0, demo_period / 2, demo_period])
snapshot_steps = np.rint(requested / dt).astype(int)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True, layout="constrained")
for ax, n in zip(axes, snapshot_steps):
    ax.plot(x, np.abs(two_state_pi[n])**2, label="Midpoint")
    ax.plot(x, np.abs(two_state_eig[n])**2, "--", label="Eigenstates")
    ax.set(xlabel="Position x", title=f"t = {t[n]:.4f}")
axes[0].set_ylabel("Probability density")
axes[-1].legend()
plt.show()

### Your turn: make D a fair test

- Use the same $(\phi_0+\phi_1)/\sqrt{2}$ initial state, signs, grid, and times for
  both methods. Keep A's Gaussian run separate.
- Derive the mean-position formula from the two phases. Which terms vanish by
  parity? Where do normalization and the energy difference enter? Use it to derive
  the two-state tunneling time.
- Evolve to the first step at or after $T=2\pi/\Delta E$. Report the maximum
  mean-position difference over **all steps with $n\Delta t\leq T$**, its time,
  and both means there.
- Overlay the densities at the nearest steps to $0,\pi/\Delta E,T$, labeling the
  methods and actual times. Identify a discrepancy pattern and one possible
  numerical cause. Propose a targeted check and predict what it would show;
  the assignment does not require another run.

Matching normalization does not prove matching dynamics: we normalized the
path-integral result at every step. The curves and snapshots test more than the norm.

## Save a practice result, then read it back

**Write real numbers and lists. Check the file you actually wrote.**

This oscillator file is for practice. Use your own double-well calculations for
the homework's `results.json`.

In [ ]:
practice_steps = np.arange(0, 129, 16)
practice = {
    "model": "harmonic oscillator practice",
    "omega": omega,
    "times": t[practice_steps].tolist(),
    "mean_position": gaussian_mean[practice_steps].tolist(),
}
practice_path = Path("lab09_practice.json")
practice_path.write_text(json.dumps(practice, indent=2, allow_nan=False))
loaded = json.loads(practice_path.read_text())
assert len(loaded["times"]) == len(loaded["mean_position"]) == 9
print("Read back:", practice_path, "with", len(loaded["times"]), "times")

## Homework export: check the indices

**33 export times do not necessarily cover a full slow oscillation.**

Use $n=0,64,128,\ldots,2048$, so the final export time is $16\pi$.

In [ ]:
export_steps = np.arange(0, 2049, 64)
export_times = export_steps * dt
assert export_steps.size == 33
assert np.isclose(export_times[-1], 16 * np.pi)
print("Export samples:", export_steps.size, "Last step:", export_steps[-1])
print("Last export time:", export_times[-1])

### Connect each JSON field to its calculation

| Fields | Source and shape |
| :--- | :--- |
| `tsn`, `alpha` | Your nine-digit TSN as text; $0.3+d/5000$ |
| `x_mean_pi`, `t_tunnel` | **A's Gaussian** run: 33 selected means; measured first major minimum |
| `E0`, `E1`, `phi0`, `phi1` | **B's original grid**: two energies; 601 real values per normalized, signed eigenvector |
| `E_S`, `E_A` | C's two Rayleigh quotients with B's original-grid matrix |
| `x_mean_eig` | D's eigenstate mean at **all 33 export times**, even beyond the plotted period |

Use `.tolist()` for NumPy arrays and `float(...)` for scalar results. Use
`json.dump(..., allow_nan=False)` to catch nonfinite values. Read the saved file
back and check lengths, finite values, norms, signs, and TSN/alpha agreement.

Keep D's path-integral comparison in the report; it does **not** belong in
`x_mean_pi`. Do not export the last convergence run's eigenvectors. Optional E
adds `delta_E_list` and `t_tunnel_list`, five values each in the specified order;
omit both fields if you skip E. This notebook does not generate homework answers.

## Work on your own version

1. **B:** get one checked eigenpair calculation working.
2. **A:** run your potential and inspect the first slow transfer.
3. **C and D:** reuse the checked matrix and propagation code.
4. **Save:** trace every exported field to the right run.

Ask me about a result you cannot explain. Bring the plot and the code that made it.

### Optional E and the checkpoint

E is bonus. At $\alpha=0.30,0.35,0.40,0.45,0.50$, compare the **measured** Gaussian
tunneling times with $\pi/\Delta E$. Compute the normalized Gaussian's fraction
in the two lowest states using the same grid inner product. Compare two cases
using that fraction, the timing difference, and any faster oscillations. Separate
evidence for higher states from numerical errors and ambiguity in the minimum.
An animation is optional and earns no points.

For Oral Checkpoint 2, practice explaining a matrix row, a normalization factor,
an initial-state choice, or an exported value. Predict a small change, make it,
and check the result. You should be able to do this with your own code and AI
assistants closed; bonus E is not required.

Before submission: restart and Run All in your own environment; inspect the plots
and saved file. Submit the PDF report to **Assignment 2**, and the source files
plus `results.json` to **Assignment 2 Code**. End the report with the syllabus's
AI-use appendix, including what you checked and what the tools got wrong, or
state that you did not use AI.